# Model Compilation

### Traditional Machine Learning Models

**Goal**: build ML models to predict Amazon prices from product descriptions. Data processing was performed in notebook `01_eda_and_preprocessing`

**Methodology**: this notebook follows an incremental baseline strategy, building a ladder of feature complexity to check whether each added layer improves performance

## Index

1. Naive baseline
2. Linear and non-linear regressors on the 5 basic features
3. High-dimensional modeling: regressors using the 5 basic features plus embeddings

In [8]:
# imports
# -------

from pathlib import Path
import pyarrow


from sklearn.kernel_approximation import Nystroem
from sklearn.linear_model import RidgeCV, Ridge, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.ensemble import RandomForestRegressor
from catboost import CatBoostRegressor
from sklearn.svm import SVR
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyRegressor

import pandas as pd
import numpy as np

import sys
sys.path.append(str(Path("../../src").resolve())) # or uv sync - but this will always work instead
from pricing_amazon_products.io import save_df
from pricing_amazon_products.experiments import (
    init_metadata_df,
    update_model_metadata, 
    evaluate_model
)
from pricing_amazon_products.config import (
    TARGET,
    N_FOLDS,
    RANDOM_STATE,
    SCORING,
    EMBEDDING_SOURCE_COL, 
    EMBEDDING_MODELS
)

import warnings
warnings.filterwarnings('ignore')

In [9]:
embedding_models_df = pd.DataFrame.from_dict(EMBEDDING_MODELS, orient="index")
embedding_models_df

,max_tokens,weight_in_million_params,dimensions,mteb_position
sentence-transformers/all-MiniLM-L6-v2,256,23,384,133
avsolatorio/GIST-small-Embedding-v0,512,33,384,50


In [4]:
# load data

split_dir = Path("../../data/splits")

y_train = pd.read_parquet(split_dir  / "y_train.parquet").drop(columns='row_id')[TARGET].values
y_test = pd.read_parquet(split_dir  / "y_test.parquet").drop(columns='row_id')[TARGET].values

X_train_base = pd.read_parquet(split_dir  / "X_train_base.parquet").drop(columns='row_id')
X_test_base = pd.read_parquet(split_dir  / "X_test_base.parquet").drop(columns='row_id')

# We are going to use the most powerful trained embedding vector
HEAVIER_EMBEDDING_MODEL = embedding_models_df["weight_in_million_params"].idxmax()
emb_name = f'{EMBEDDING_SOURCE_COL}__{HEAVIER_EMBEDDING_MODEL.replace('/', '_')}'
X_train_embeddings = pd.read_parquet(split_dir  / f"X_train_with_embeddings__{emb_name}.parquet").drop(columns='row_id')
X_test_embeddings = pd.read_parquet(split_dir  / f"X_test_with_embeddings__{emb_name}.parquet").drop(columns='row_id')


In [18]:
# Size

print("Train features shape:", X_train_base.shape)
print("Train embeddings + features shape:", X_train_embeddings.shape)
print("Train target shape:", y_train.shape)
print("Test features shape:", X_test_embeddings.shape)
print("Test embeddings + target shape:", y_train.shape)
print("Test target shape:", y_test.shape)

Train features shape: (59919, 5)
Train embeddings + features shape: (59919, 389)
Train target shape: (59919,)
Test features shape: (14980, 389)
Test embeddings + target shape: (59919,)
Test target shape: (14980,)


In [19]:
X_train_base.head(2)

,log_value,unit_grouped_fluid_ounce,unit_grouped_none,unit_grouped_other,unit_grouped_ounce
64402,3.650658,0,0,0,1
22256,3.496508,0,0,0,1


In [20]:
X_train_embeddings.head(2)

,log_value,unit_grouped_fluid_ounce,unit_grouped_none,unit_grouped_other,unit_grouped_ounce,item_name_emb_0,item_name_emb_1,item_name_emb_2,item_name_emb_3,item_name_emb_4,...,item_name_emb_374,item_name_emb_375,item_name_emb_376,item_name_emb_377,item_name_emb_378,item_name_emb_379,item_name_emb_380,item_name_emb_381,item_name_emb_382,item_name_emb_383
0,3.650658,0,0,0,1,-0.040625,-0.020483,-0.004148,0.015989,-0.003517,...,0.016781,-0.086198,0.021473,-0.040283,0.006972,0.004834,-0.000212,0.023138,0.068493,-0.007558
1,3.496508,0,0,0,1,-0.044942,-0.039738,0.014793,-0.042243,0.031738,...,0.007900,0.044940,-0.008965,-0.054913,0.016335,0.031158,-0.004096,-0.061639,0.061490,-0.052469


## Configuration

----

Global settings to keep the experiments reproducible and easy to update:

- `FORCE_OVERWRITE`: if `True`, saved results are always regenerated.
- `OVERWRITE_IF_BETTER_CV`: if `True`, an existing result is replaced only when the new model achieves a better cross-validated score.

In [21]:
# -------------------------------------------------------------------
# Configuration
# -------------------------------------------------------------------

FORCE_OVERWRITE = False
OVERWRITE_IF_BETTER_CV = True

**Model target and evaluator**

- Target: `log_price`
- Metric: RMSE, to be minimized
- Note: Since the target is `log(price)`, RMSE will evaluates error in log space

**Metadata**

This notebook keeps a small experiment log with one row per model run.  
Each row stores the model name, model family, feature label, target, hyperparameters, and evaluation scores so results can be compared later.

The stored metrics will be replaced for a given model if `FORCE_OVERWRITE` is enabled, or if `OVERWRITE_IF_BETTER_CV` is enabled and the new cross-validated score is better than the one already stored.

In [22]:
results_metadata_df = init_metadata_df(overwrite=False)
results_metadata_df

Skipping model_metadata.parquet (already exists and overwrite mode set to False)


,model_name,model_family,feature_label,target,scoring_name,params,cv_score_mean,cv_score_std,train_score,test_score
0,Constant Median,Dummy,Naive (constant baseline),log_price,neg_root_mean_squared_error,"{""constant"": null, ""quantile"": null, ""strategy...",1.040703,0.004772,1.040656,1.039322
1,Linear Regression,Linear,Baseline on basic features,log_price,neg_root_mean_squared_error,"{""copy_X"": true, ""fit_intercept"": true, ""n_job...",1.008901,0.004754,1.008771,1.009703
2,Random Forest,Tree-based,Baseline on basic features,log_price,neg_root_mean_squared_error,"{""bootstrap"": true, ""ccp_alpha"": 0.0, ""criteri...",0.970164,0.004397,0.953868,0.964460
3,Ridge (L2),Linear,High dimensional,log_price,neg_root_mean_squared_error,"{""memory"": null, ""steps"": [[""scaler"", ""Standar...",0.880379,0.002973,0.874106,0.872967
4,LightGBM,Tree-based,High dimensional,log_price,neg_root_mean_squared_error,"{""boosting_type"": ""gbdt"", ""class_weight"": null...",0.888764,0.003283,0.864097,0.885090
5,CatBoost,Tree-based,High dimensional,log_price,neg_root_mean_squared_error,"{""iterations"": 150, ""learning_rate"": 0.05, ""de...",0.906867,0.003560,0.896892,0.903251
6,MLP,NN,High dimensional,log_price,neg_root_mean_squared_error,"{""memory"": null, ""steps"": [[""scaler"", ""Standar...",0.827764,0.003231,0.620621,0.803746
7,Nystroem + Ridge,Non-Parametric,High dimensional,log_price,neg_root_mean_squared_error,"{""memory"": null, ""steps"": [[""standardscaler"", ...",0.908836,0.004388,0.901868,0.900014


## 1 | Naive baseline

----


This baseline predicts a single **constant value: the median of the training target**. 

No features are used, so this is a simple benchmark that ignores all product characteristics.

The RMSE of this naive predictor is the baseline, a minimum floor to beat.

**A) Median baseline**

In [12]:
dm = DummyRegressor(strategy="median")

dm_model, dm_results = evaluate_model(
    dm,
    X_train_base,
    y_train,
    X_test_base,
    y_test,
    feature_label='Naive (constant baseline)',
    model_name='Constant Median',
    model_family='Dummy'
)
dm_results

model_name                                         Constant Median
model_family                                                 Dummy
feature_label                            Naive (constant baseline)
target                                                   log_price
scoring_name                           neg_root_mean_squared_error
params           {"constant": null, "quantile": null, "strategy...
cv_score_mean                                             1.040703
cv_score_std                                              0.004772
train_score                                               1.040656
test_score                                                1.039322
dtype: object

In [9]:
results_metadata_df = update_model_metadata(
    results_metadata_df, 
    dm_results, 
    FORCE_OVERWRITE, 
    OVERWRITE_IF_BETTER_CV
)

Saved model_metadata.parquet


**Observations**

- The naive median baseline is a strong sanity check, providing a simple baseline with RMSE around `1.04` (training cross val score)

## 2 | Baseline on basic features

----

We train two baseline models using only the 5 basic features, without embeddings:
- Linear Regression, as a simple linear benchmark
- Random Forest, as a non-linear benchmark

These models also establish the minimum performance threshold that the embeddings must beat to justify their storage and computational cost

**B) Linear Regression (no embeddings)**

In [10]:
lr = LinearRegression(n_jobs=-1)

lr_model, lr_results = evaluate_model(
    lr,
    X_train_base,
    y_train,
    X_test_base,
    y_test,
    feature_label='Classical (basic features)',
    model_name='Linear Regression',
    model_family='Linear'
)
lr_results

model_name                                       Linear Regression
model_family                                                Linear
feature_label                           Baseline on basic features
target                                                   log_price
scoring_name                           neg_root_mean_squared_error
params           {"copy_X": true, "fit_intercept": true, "n_job...
cv_score_mean                                             1.008901
cv_score_std                                              0.004754
train_score                                               1.008771
test_score                                                1.009703
dtype: object

In [11]:
results_metadata_df = update_model_metadata(
    results_metadata_df, 
    lr_results, 
    FORCE_OVERWRITE, 
    OVERWRITE_IF_BETTER_CV
)

Saved model_metadata.parquet


**C) Random Forest (no embeddings)**

In [12]:
rf = RandomForestRegressor(max_depth=10, random_state=RANDOM_STATE, n_jobs=-1)

rf_model, rf_results = evaluate_model(
    rf,
    X_train_base,
    y_train,
    X_test_base,
    y_test,
    feature_label='Classical (basic features)',
    model_name='Random Forest',
    model_family='Tree-based'
)
rf_results

model_name                                           Random Forest
model_family                                            Tree-based
feature_label                           Baseline on basic features
target                                                   log_price
scoring_name                           neg_root_mean_squared_error
params           {"bootstrap": true, "ccp_alpha": 0.0, "criteri...
cv_score_mean                                             0.970164
cv_score_std                                              0.004397
train_score                                               0.953868
test_score                                                 0.96446
dtype: object

In [13]:
results_metadata_df = update_model_metadata(
    results_metadata_df, 
    rf_results, 
    FORCE_OVERWRITE, 
    OVERWRITE_IF_BETTER_CV
)

Saved model_metadata.parquet


**Observations**

- Linear Regression improves slightly over the naive baseline, confirming that the basic engineered features contain useful signal
- Random Forest performs better than Linear Regression on cross validation, suggesting that some non-linear interactions are present in the data
- These results justify moving on to embedding-based feature sets to test whether richer text representations can further improve performance

## 3 | High-dimensional modeling

----

In this stage, we evaluate whether richer text representations can improve predictive performance beyond the basic structured features.

We build models using the 5 basic features + high-dimensional embedding features derived from product description. Therefore, it tests whether semantic information captured by embeddings provides a meaningful gain over the simpler baseline models, and whether the additional storage and computation cost is justified.

**Models to be Explored that are expected to work well on semantic vector embeddings**

| Model category| Models | Reason |
|---------------------|---------------------|---------------------|
| **I. Linear Regressors**         | Ridge Regression| Keeps all embedding dimensions and distributes coefficient weights smoothly across highly correlated semantic variables |
| **II. Decision Tree Based Regressors**  | LightGBM, CatBoost | Natively handles non-linear feature splits and high dimensions efficiently |
| **III. Non-parametric Regressors** | Support Vector Regressor| Handles non-linear relationship thanks to the RBF kernel (the model is distance based, as embeddings) |
| **IV. Neural Network Regression** | MLPRegressor| Industry standard for learning dense cross-feature representations |



### __I | Linear models__



__D | Ridge regressor__


In [14]:
alphas = [0.1, 1.0, 10.0, 50.0, 100.0, 1000.0]

ridge_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge_cv", RidgeCV(
        alphas=alphas,
        scoring=SCORING,
        cv=N_FOLDS
    ))
])

ridge_model, ridge_results = evaluate_model(
    ridge_pipeline,
    X_train_embeddings,
    y_train,
    X_test_embeddings,
    y_test,
    feature_label='Classical (basic features and embeddings)',
    model_name='Ridge (L2)',
    model_family='Linear'
)
ridge_results

model_name                                              Ridge (L2)
model_family                                                Linear
feature_label                                     High dimensional
target                                                   log_price
scoring_name                           neg_root_mean_squared_error
params           {"memory": null, "steps": [["scaler", "Standar...
cv_score_mean                                             0.880379
cv_score_std                                              0.002973
train_score                                               0.874106
test_score                                                0.872967
dtype: object

In [15]:
optimal_alpha = ridge_model.named_steps["ridge_cv"].alpha_

print(f"Optimal alpha: {optimal_alpha:.4f}")

Optimal alpha: 50.0000


In [16]:
results_metadata_df = update_model_metadata(
    results_metadata_df, 
    ridge_results, 
    FORCE_OVERWRITE, 
    OVERWRITE_IF_BETTER_CV
)

Saved model_metadata.parquet


In [17]:
results_metadata_df

,model_name,model_family,feature_label,target,scoring_name,params,cv_score_mean,cv_score_std,train_score,test_score
0,Constant Median,Dummy,Naive (constant baseline),log_price,neg_root_mean_squared_error,"{""constant"": null, ""quantile"": null, ""strategy...",1.040703,0.004772,1.040656,1.039322
1,Linear Regression,Linear,Baseline on basic features,log_price,neg_root_mean_squared_error,"{""copy_X"": true, ""fit_intercept"": true, ""n_job...",1.008901,0.004754,1.008771,1.009703
2,Random Forest,Tree-based,Baseline on basic features,log_price,neg_root_mean_squared_error,"{""bootstrap"": true, ""ccp_alpha"": 0.0, ""criteri...",0.970164,0.004397,0.953868,0.96446
3,Ridge (L2),Linear,High dimensional,log_price,neg_root_mean_squared_error,"{""memory"": null, ""steps"": [[""scaler"", ""Standar...",0.880379,0.002973,0.874106,0.872967


### __II | Decision-trees' based models__



__E | LightGBM__


In [18]:
lgbm = LGBMRegressor(
    max_depth=5,
    num_leaves=20,
    learning_rate=0.05, 
    random_state=RANDOM_STATE, 
    n_jobs=-1
)

lgbm_model, lgbm_results = evaluate_model(
    lgbm,
    X_train_embeddings,
    y_train,
    X_test_embeddings,
    y_test,
    feature_label='Classical (basic features and embeddings)',
    model_name='LightGBM',
    model_family='Tree-based'
)
lgbm_results

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.121698 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 98183
[LightGBM] [Info] Number of data points in the train set: 47935, number of used features: 389
[LightGBM] [Info] Start training from score 2.633361
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.118386 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 98183
[LightGBM] [Info] Number of data points in the train set: 47935, number of used features: 389
[LightGBM] [Info] Start training from score 2.633146
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.128455 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 98183
[LightGBM] [Info] Number of data points in the train set: 47935, number of used features: 389
[LightGBM] [Info] Sta

model_name                                                LightGBM
model_family                                            Tree-based
feature_label                                     High dimensional
target                                                   log_price
scoring_name                           neg_root_mean_squared_error
params           {"boosting_type": "gbdt", "class_weight": null...
cv_score_mean                                             0.888764
cv_score_std                                              0.003283
train_score                                               0.864097
test_score                                                 0.88509
dtype: object

In [19]:
results_metadata_df = update_model_metadata(
    results_metadata_df, 
    lgbm_results, 
    FORCE_OVERWRITE, 
    OVERWRITE_IF_BETTER_CV
)

Saved model_metadata.parquet


__F | CatBoost__

In [20]:
cm = CatBoostRegressor(
    depth=5,
    l2_leaf_reg=10.0,
    learning_rate=0.05, 
    iterations=150,
    random_seed=RANDOM_STATE,
    verbose=0
)

cm_model, cm_results = evaluate_model(
    cm,
    X_train_embeddings,
    y_train,
    X_test_embeddings,
    y_test,
    feature_label='Classical (basic features and embeddings)',
    model_name='CatBoost',
    model_family='Tree-based'
)
cm_results

model_name                                                CatBoost
model_family                                            Tree-based
feature_label                                     High dimensional
target                                                   log_price
scoring_name                           neg_root_mean_squared_error
params           {"iterations": 150, "learning_rate": 0.05, "de...
cv_score_mean                                             0.906867
cv_score_std                                               0.00356
train_score                                               0.896892
test_score                                                0.903251
dtype: object

In [21]:
results_metadata_df = update_model_metadata(
    results_metadata_df, 
    cm_results, 
    FORCE_OVERWRITE, 
    OVERWRITE_IF_BETTER_CV
)

Saved model_metadata.parquet


### **III. Non-parametric Regressors**

**G | SVR**

In [17]:

# Standard SVR with an RBF kernel scales quadratically, making it extremely computationally expensive for this datasets (~75K rows).
# SVR stands out with embedding vector as a predictor bc of its RBK kernel which is based on distances as the semantic vectors
# Nystroem + Ridge is a lighter alternative by approximating the BF kernel map -> retaining distance based learning

fast_rbf_model = make_pipeline(
    StandardScaler(),
    Nystroem(kernel='rbf', n_components=1000, random_state=RANDOM_STATE),
    Ridge(alpha=ridge_model.named_steps["ridge_cv"].alpha_)
)

rbf_model, rbf_results = evaluate_model(
    fast_rbf_model,
    X_train_embeddings,
    y_train,
    X_test_embeddings,
    y_test,
    feature_label='Classical (basic features and embeddings)',
    model_name='Nystroem + Ridge',
    model_family='Non-Parametric'
)
rbf_results

model_name                                        Nystroem + Ridge
model_family                                        Non-Parametric
feature_label                                     High dimensional
target                                                   log_price
scoring_name                           neg_root_mean_squared_error
params           {"memory": null, "steps": [["standardscaler", ...
cv_score_mean                                             0.908836
cv_score_std                                              0.004388
train_score                                               0.901868
test_score                                                0.900014
dtype: object

In [18]:
results_metadata_df = update_model_metadata(
    results_metadata_df, 
    rbf_results, 
    FORCE_OVERWRITE, 
    OVERWRITE_IF_BETTER_CV
)

Saved model_metadata.parquet


### **IV. Neural Network Regressor**

**H | MLP**

In [9]:
mlp = MLPRegressor(
    hidden_layer_sizes=(128, 64),
    activation="relu",
    alpha=0.1,
    batch_size=256,
    max_iter=300,
    early_stopping=True,
    random_state=RANDOM_STATE
)

mlp_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", mlp)
])

mlp_model, mlp_results = evaluate_model(
    mlp_pipeline,
    X_train_embeddings,
    y_train,
    X_test_embeddings,
    y_test,
    feature_label='Classical (basic features and embeddings)',
    model_name='MLP',
    model_family='NN'
)
mlp_results

model_name                                                     MLP
model_family                                                    NN
feature_label                                     High dimensional
target                                                   log_price
scoring_name                           neg_root_mean_squared_error
params           {"memory": null, "steps": [["scaler", "Standar...
cv_score_mean                                             0.827764
cv_score_std                                              0.003231
train_score                                               0.620621
test_score                                                0.803746
dtype: object

In [10]:
results_metadata_df = update_model_metadata(
    results_metadata_df, 
    mlp_results, 
    FORCE_OVERWRITE, 
    OVERWRITE_IF_BETTER_CV
)

Saved model_metadata.parquet
